# Add-it CityPersons SD3.5 Medium

Training-free pedestrian insertion using the Add-it-style pipeline in `/kaggle/working/VIN/addit(experimental)/`.


## 1. Install Dependencies


In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors ultralytics huggingface_hub scikit-image


## 2. Clone Or Update Repo


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/VIN.git"
REPO_DIR = Path("/kaggle/working/VIN")

if REPO_DIR.exists():
    %cd /kaggle/working/VIN
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/VIN

PROJECT_DIR = REPO_DIR
ADDIT_DIR = PROJECT_DIR / "addit(experimental)"
if not (PROJECT_DIR / "sd35_config.py").exists() or not ADDIT_DIR.exists():
    raise FileNotFoundError(f"Expected repo root with sd35_config.py and addit(experimental)/: {PROJECT_DIR}")

%cd {PROJECT_DIR}
print("Project dir:", PROJECT_DIR)
!git log --oneline -3
!grep -n "ADDIT_FINAL_PERSON_CUTOUT\|ADDIT_WEIGHTED_EXTENDED_ATTENTION" "addit(experimental)/addit_config.py"


## 3. Imports


In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/VIN") if Path("/kaggle/working/VIN/addit(experimental)").exists() else Path.cwd()
ADDIT_DIR = PROJECT_DIR / "addit(experimental)"
%cd {PROJECT_DIR}
for module_dir in (PROJECT_DIR, ADDIT_DIR):
    module_dir = str(module_dir)
    if module_dir in sys.path:
        sys.path.remove(module_dir)
    sys.path.insert(0, module_dir)

for stale_module in ("addit_config", "addit_core", "addit_pipeline"):
    sys.modules.pop(stale_module, None)

from sd35_config import *
from sd35_data import load_records, preview_prompt_samples
import sd35_model as sd35_model_module
from sd35_model import build_img2img_pipeline
from sd35_utils import clear_cuda
from addit_config import *
import inspect
import addit_core as addit_core_module
import addit_pipeline as addit_pipeline_module
from addit_pipeline import AddItCityPersonsPipeline

print("Imports OK")
print("addit_config:", sys.modules["addit_config"].__file__)
print("addit_core:", addit_core_module.__file__)
print("transfer_noise_structure:", inspect.signature(addit_core_module.transfer_noise_structure))
print("addit_pipeline:", addit_pipeline_module.__file__)
print("ADDIT_WEIGHTED_EXTENDED_ATTENTION:", ADDIT_WEIGHTED_EXTENDED_ATTENTION)
print("ADDIT_FINAL_PERSON_CUTOUT:", ADDIT_FINAL_PERSON_CUTOUT)
print("ADDIT_FALLBACK_TO_NATIVE_IMG2IMG:", ADDIT_FALLBACK_TO_NATIVE_IMG2IMG)


## 4. Runtime Check


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(idx, props.name, round(props.total_memory / 1024**3, 2), "GB")
print("Resolution:", RESOLUTION)
print("Add-it output:", ADDIT_OUTPUT_DIR)


## 5. Hugging Face Login


In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = hf_token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("Kaggle secret lookup skipped:", type(exc).__name__)

if hf_token:
    login(token=hf_token)
    print("HF login OK")
else:
    print("No HF_TOKEN found; continue only if model is already accessible.")


## 6. Load Models


In [ ]:
import torch

primary_device = globals().get("ADDIT_PRIMARY_DEVICE", TRAIN_DEVICE) if torch.cuda.is_available() else "cpu"

use_two_gpus = bool(globals().get("ADDIT_USE_TWO_GPUS", True))
generation_transformer_device = globals().get("ADDIT_TRANSFORMER_DEVICE", "cuda:1")

if use_two_gpus and torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    sd35_model_module.USE_MODEL_CPU_OFFLOAD = False

pipe = build_img2img_pipeline(device=primary_device)
if use_two_gpus and torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    print("Add-it generation-only 2-GPU mode: load on", primary_device, "| denoise transformer temporarily on", generation_transformer_device)
else:
    print("Add-it single-device mode:", primary_device)

device = primary_device
addit_pipe = AddItCityPersonsPipeline(pipe, device=device)
print("Add-it pipeline default transformer device:", addit_pipe.device)
print("Add-it pipeline VAE device:", addit_pipe.vae_device)
print("Add-it pipeline prompt device:", addit_pipe.prompt_device)


## 7. Dataset Scan


In [ ]:
records = load_records()
print("records:", len(records))
preview_prompt_samples(records, n=2)


## 8. Demo Single Image


In [ ]:
demo_record = records[0]
demo_variant = "add_single_pedestrian"
demo = addit_pipe.run_single(demo_record, demo_variant, seed=ADDIT_SEED, device=device)
print(demo)
if demo.source_image is not None and demo.result_image is not None:
    display(demo.source_image)
    display(demo.result_image)
    print("debug:", demo.debug_path)


## 9. Smoke Run And Summary


In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim

def background_ssim_outside_bbox(source, result, bbox):
    src = np.asarray(source.convert("RGB"), dtype=np.uint8).copy()
    res = np.asarray(result.convert("RGB"), dtype=np.uint8).copy()
    if bbox is not None:
        x1, y1, x2, y2 = [int(round(v)) for v in bbox]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(src.shape[1], x2), min(src.shape[0], y2)
        src[y1:y2, x1:x2] = 0
        res[y1:y2, x1:x2] = 0
    return float(ssim(src, res, channel_axis=2, data_range=255))

smoke_records = records[:5]
smoke_variants = [
    "add_single_pedestrian",
    "add_two_pedestrians",
    "add_small_group",
    "add_occluded_pedestrian",
    "add_distant_pedestrian",
]
results = addit_pipe.run_batch(
    smoke_records,
    variants=smoke_variants,
    device=device,
    max_images=len(smoke_records),
    seed=ADDIT_SEED,
)
accepted = sum(1 for item in results if item.success)
print("Accepted:", accepted, "/", len(results))
for item in results:
    bg_ssim = None
    if item.source_image is not None and item.result_image is not None:
        bg_ssim = background_ssim_outside_bbox(item.source_image, item.result_image, item.insert_bbox)
    print(item.variant, item.success, item.reject_reason, "bg_ssim=", bg_ssim, item.debug_path)

clear_cuda()


## 10. Export Outputs


In [ ]:
# Run after generation if you want a zip artifact.
from pathlib import Path
import shutil

output_dir = Path(ADDIT_OUTPUT_DIR)
zip_base = Path('/kaggle/working') / output_dir.name
zip_path = zip_base.with_suffix('.zip')

if not output_dir.exists():
    raise FileNotFoundError(f"Output directory not found: {output_dir}")
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_base), 'zip', root_dir=output_dir)
print('Saved zip:', zip_path)
print('Size MB:', round(zip_path.stat().st_size / 1024**2, 2))
